## U24AI038 NLP ASSIGNMENT 4 AND 5

In [ ]:
from pathlib import Path
data = Path('../assignment 1/tokenized_words.txt')

from collections import defaultdict

unigrams = defaultdict(int)
bigrams = defaultdict(int)
trigrams = defaultdict(int)
quadgrams = defaultdict(int)

dev = []
test = []

with data.open('r', encoding='utf') as f:
    for i, line in enumerate(f):
        line = line.strip()

        if i < 1_000_000:
            curr = line.split()

            bi_line = ["<s>"] + curr + ["</s>"]
            tri_line = ["<s>", "<s>"] + curr + ["</s>"]
            quad_line = ["<s>", "<s>", "<s>"] + curr + ["</s>"]

            bi = zip(bi_line, bi_line[1:])
            tri = zip(tri_line, tri_line[1:], tri_line[2:])
            quad = zip(quad_line, quad_line[1:], quad_line[2:], quad_line[3:])

            for word in curr:
                unigrams[word] += 1

            for bigram in bi:
                bigrams[bigram] += 1

            for trigram in tri:
                trigrams[trigram] += 1

            for quadgram in quad:
                quadgrams[quadgram] += 1

        elif i < 1_001_000:
            test.append(line)

        elif i < 1_002_000:
            dev.append(line)

        else:
            break

print("Unigrams:", len(unigrams))
print("Bigrams:", len(bigrams))
print("Trigrams:", len(trigrams))
print("Quadgrams:", len(quadgrams))
print("Dev:", len(dev))
print("Test:", len(test))

Unigrams: 755566
Bigrams: 6004138
Trigrams: 10834281
Quadgrams: 13060259
Dev: 1000
Test: 1000


In [9]:
def unigram_model(sentence):
    words = sentence.split()
    log_prob = 0.0

    for word in words:
        count = unigrams[word]

        if count == 0:
            return -math.inf

        log_prob += math.log(count / N)

    return log_prob


def bigram_model(sentence):
    words = ["<s>"] + sentence.split() + ["</s>"]
    log_prob = 0.0

    for i in range(1, len(words)):
        w1 = words[i - 1]
        w2 = words[i]

        count = bigrams[(w1, w2)]

        if count == 0:
            return -math.inf

        if w1 == "<s>":
            context_count = TRAIN_SIZE
        else:
            context_count = unigrams[w1]

        log_prob += math.log(count / context_count)

    return log_prob


def trigram_model(sentence):
    words = ["<s>", "<s>"] + sentence.split() + ["</s>"]
    log_prob = 0.0

    for i in range(2, len(words)):
        w1 = words[i - 2]
        w2 = words[i - 1]
        w3 = words[i]

        count = trigrams[(w1, w2, w3)]
        context_count = bigrams[(w1, w2)]

        if count == 0:
            return -math.inf

        # This also prevents division by zero
        if context_count == 0:
            return -math.inf

        log_prob += math.log(count / context_count)

    return log_prob


def quadgram_model(sentence):
    words = ["<s>", "<s>", "<s>"] + sentence.split() + ["</s>"]
    log_prob = 0.0

    for i in range(3, len(words)):
        w1 = words[i - 3]
        w2 = words[i - 2]
        w3 = words[i - 1]
        w4 = words[i]

        count = quadgrams[(w1, w2, w3, w4)]
        context_count = trigrams[(w1, w2, w3)]

        if count == 0:
            return -math.inf

        if context_count == 0:
            return -math.inf

        log_prob += math.log(count / context_count)

    return log_prob

In [10]:
test_log_prob_uni = 0
test_log_prob_bi = 0
test_log_prob_tri = 0
test_log_prob_quad = 0

for sentence in test:
    test_log_prob_uni += unigram_model(sentence)
    test_log_prob_bi += bigram_model(sentence)
    test_log_prob_tri += trigram_model(sentence)
    test_log_prob_quad += quadgram_model(sentence)

print("Unigram:", test_log_prob_uni)
print("Bigram:", test_log_prob_bi)
print("Trigram:", test_log_prob_tri)
print("Quadrigram:", test_log_prob_quad)

Unigram: -inf
Bigram: -inf
Trigram: -inf
Quadrigram: -inf


### Add-1 Smoothing

In [11]:
def unigram_add1(sentence):
    words = sentence.split()
    log_prob = 0.0

    for word in words:
        count = unigrams[word]

        prob = (count + 1) / (N + V)

        log_prob += math.log(prob)

    return log_prob


def bigram_add1(sentence):
    words = ["<s>"] + sentence.split() + ["</s>"]
    log_prob = 0.0

    for i in range(1, len(words)):
        w1 = words[i - 1]
        w2 = words[i]

        count = bigrams[(w1, w2)]

        if w1 == "<s>":
            context_count = TRAIN_SIZE
        else:
            context_count = unigrams[w1]

        prob = (count + 1) / (context_count + V)

        log_prob += math.log(prob)

    return log_prob


def trigram_add1(sentence):
    words = ["<s>", "<s>"] + sentence.split() + ["</s>"]
    log_prob = 0.0

    for i in range(2, len(words)):
        w1 = words[i - 2]
        w2 = words[i - 1]
        w3 = words[i]

        count = trigrams[(w1, w2, w3)]
        context_count = bigrams[(w1, w2)]

        prob = (count + 1) / (context_count + V)

        log_prob += math.log(prob)

    return log_prob


def quadgram_add1(sentence):
    words = ["<s>", "<s>", "<s>"] + sentence.split() + ["</s>"]
    log_prob = 0.0

    for i in range(3, len(words)):
        w1 = words[i - 3]
        w2 = words[i - 2]
        w3 = words[i - 1]
        w4 = words[i]

        count = quadgrams[(w1, w2, w3, w4)]
        context_count = trigrams[(w1, w2, w3)]

        prob = (count + 1) / (context_count + V)

        log_prob += math.log(prob)

    return log_prob

In [12]:
test_log_prob_uni = 0
test_log_prob_bi = 0
test_log_prob_tri = 0
test_log_prob_quad = 0

for sentence in test:
    test_log_prob_uni += unigram_add1(sentence)
    test_log_prob_bi += bigram_add1(sentence)
    test_log_prob_tri += trigram_add1(sentence)
    test_log_prob_quad += quadgram_add1(sentence)

print("Unigram:", test_log_prob_uni)
print("Bigram:", test_log_prob_bi)
print("Trigram:", test_log_prob_tri)
print("Quadrigram:", test_log_prob_quad)

Unigram: -135531.52640673518
Bigram: -162883.5517659955
Trigram: -192278.98560599858
Quadrigram: -205451.35187584907


In [13]:
total_words = sum(len(sentence.split()) for sentence in test)

uni_pp = math.exp(-test_log_prob_uni / total_words)

bi_tokens = sum(max(0, len(sentence.split()) - 1) for sentence in test)
bi_pp = math.exp(-test_log_prob_bi / bi_tokens)

tri_tokens = sum(max(0, len(sentence.split()) - 2) for sentence in test)
tri_pp = math.exp(-test_log_prob_tri / tri_tokens)

quad_tokens = sum(max(0, len(sentence.split()) - 3) for sentence in test)
quad_pp = math.exp(-test_log_prob_quad / quad_tokens)

print("Unigram PP:", uni_pp)
print("Bigram PP:", bi_pp)
print("Trigram PP:", tri_pp)
print("Quadrigram PP:", quad_pp)

Unigram PP: 7164.085018103186
Bigram PP: 90111.94380796659
Trigram PP: 1926492.1725443518
Quadrigram PP: 17881709.69783517


In [14]:
k = 0.3


def unigram_addk(sentence):
    words = sentence.split()
    log_prob = 0.0

    for word in words:
        count = unigrams[word]

        prob = (count + k) / (N + k * V)

        log_prob += math.log(prob)

    return log_prob


def bigram_addk(sentence):
    words = ["<s>"] + sentence.split() + ["</s>"]
    log_prob = 0.0

    for i in range(1, len(words)):
        w1 = words[i - 1]
        w2 = words[i]

        count = bigrams[(w1, w2)]

        if w1 == "<s>":
            context_count = TRAIN_SIZE
        else:
            context_count = unigrams[w1]

        prob = (count + k) / (context_count + k * V)

        log_prob += math.log(prob)

    return log_prob


def trigram_addk(sentence):
    words = ["<s>", "<s>"] + sentence.split() + ["</s>"]
    log_prob = 0.0

    for i in range(2, len(words)):
        w1 = words[i - 2]
        w2 = words[i - 1]
        w3 = words[i]

        count = trigrams[(w1, w2, w3)]
        context_count = bigrams[(w1, w2)]

        prob = (count + k) / (context_count + k * V)

        log_prob += math.log(prob)

    return log_prob


def quadgram_addk(sentence):
    words = ["<s>", "<s>", "<s>"] + sentence.split() + ["</s>"]
    log_prob = 0.0

    for i in range(3, len(words)):
        w1 = words[i - 3]
        w2 = words[i - 2]
        w3 = words[i - 1]
        w4 = words[i]

        count = quadgrams[(w1, w2, w3, w4)]
        context_count = trigrams[(w1, w2, w3)]

        prob = (count + k) / (context_count + k * V)

        log_prob += math.log(prob)

    return log_prob

In [15]:
test_log_prob_uni = 0
test_log_prob_bi = 0
test_log_prob_tri = 0
test_log_prob_quad = 0

for sentence in test:
    test_log_prob_uni += unigram_addk(sentence)
    test_log_prob_bi += bigram_addk(sentence)
    test_log_prob_tri += trigram_addk(sentence)
    test_log_prob_quad += quadgram_addk(sentence)

print("Unigram:", test_log_prob_uni)
print("Bigram:", test_log_prob_bi)
print("Trigram:", test_log_prob_tri)
print("Quadrigram:", test_log_prob_quad)

Unigram: -135859.3082489144
Bigram: -152677.34056098427
Trigram: -185676.44542842705
Quadrigram: -201400.6033312104


In [16]:
total_words = sum(len(sentence.split()) for sentence in test)

uni_pp = math.exp(-test_log_prob_uni / total_words)

bi_tokens = sum(max(0, len(sentence.split()) - 1) for sentence in test)
bi_pp = math.exp(-test_log_prob_bi / bi_tokens)

tri_tokens = sum(max(0, len(sentence.split()) - 2) for sentence in test)
tri_pp = math.exp(-test_log_prob_tri / tri_tokens)

quad_tokens = sum(max(0, len(sentence.split()) - 3) for sentence in test)
quad_pp = math.exp(-test_log_prob_quad / quad_tokens)

print("Unigram PP:", uni_pp)
print("Bigram PP:", bi_pp)
print("Trigram PP:", tri_pp)
print("Quadrigram PP:", quad_pp)

Unigram PP: 7319.550382243311
Bigram PP: 44087.74540848576
Trigram PP: 1172084.1877104957
Quadrigram PP: 12865245.656309213


## Final Results
> ### Add-1
>- Unigram PP: 7164.085018103186
>- Bigram PP: 90111.94380796659
>- Trigram PP: 1926492.1725443518
>- Quadrigram PP: 17881709.69783517

> ### Add-k
>- Unigram PP: 7319.550382243311
>- Bigram PP: 44087.74540848576
>- Trigram PP: 1172084.1877104957
>- Quadrigram PP: 12865245.656309213